# NB03b — MGnify geographic holdout evaluation (Exploratory)

**Status:** Exploratory analysis.

**Goal:** Test geographic generalization on MGnify dataset, mirroring NB03 for SPIRE.

**Design:** Train on non-Australia MAGs; predict on Australia. Same region definition as NB03 (latitude −10° to −45°, longitude 110° to 155°).

**Output:** `data/mgnify_geographic_holdout.csv`.


In [1]:
print("NB03b executing — geographic holdout evaluation on MGnify (exploratory).")

NB03b executing — geographic holdout evaluation on MGnify (exploratory).


In [2]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / 'scripts'))
from modelling import MAG_DENSITY_FEATURES, geographic_holdout_eval
from evaluation import holdout_vs_cv_ratio

sys.path.insert(0, '/home/hmacgregor/BERIL-research-observatory/tools')
from figure_style import apply_style, save, FIGW, ROW_H, PALETTE, grid_h
apply_style()

DATA_DIR = Path.cwd().parent / 'data'
FIG_DIR = Path.cwd().parent / 'figures'
FIG_DIR.mkdir(exist_ok=True)

In [3]:
df = pd.read_csv(DATA_DIR / 'mgnify_mag_feature_matrix.csv')
TARGET = 'PF1_Cu'

# Reconstruct full MAG density feature list
subcat_density_cols = [c for c in df.columns if c.startswith('ko_per_mb_') and c != 'ko_per_mb_primary']
MAG_ALL_DENSITY = MAG_DENSITY_FEATURES + subcat_density_cols

# Load in-distribution CV RMSE for M1 from NB02b
cv_results = pd.read_csv(DATA_DIR / 'mgnify_mobility_prediction_results.csv')
if 'M1' in cv_results['model'].values:
    m1_cv_rmse = cv_results[cv_results['model'] == 'M1']['rmse'].mean()
else:
    m1_cv_rmse = np.nan
    print("Warning: M1 results not found in cv_results.")

print(f"M1 in-distribution CV RMSE: {m1_cv_rmse:.4f}")
print(f"MAG density features for holdout: {MAG_ALL_DENSITY}")

M1 in-distribution CV RMSE: 0.0385
MAG density features for holdout: ['ko_per_mb_primary', 'ko_per_mb_resistance', 'ko_per_mb_transport', 'ko_per_mb_sensing', 'ko_per_mb_metabolism', 'ko_per_mb_cofactor']


In [4]:
# Holdout region: Australia
AUSTRALIA_LAT = (-45, -10)
AUSTRALIA_LON = (110, 155)

holdout_mask = (
    (df['latitude'] >= AUSTRALIA_LAT[0]) & (df['latitude'] <= AUSTRALIA_LAT[1]) &
    (df['longitude'] >= AUSTRALIA_LON[0]) & (df['longitude'] <= AUSTRALIA_LON[1])
).values

# Require complete feature/target rows
required_cols = [TARGET] + MAG_ALL_DENSITY
valid_mask = df[[c for c in required_cols if c in df.columns]].notna().all(axis=1).values

holdout_mask_valid = holdout_mask & valid_mask
print(f"Australia MAGs (valid): {holdout_mask_valid.sum()}")
print(f"Training MAGs (valid): {(~holdout_mask & valid_mask).sum()}")

Australia MAGs (valid): 97
Training MAGs (valid): 7876


In [5]:
if holdout_mask_valid.sum() < 10:
    print(f"WARNING: fewer than 10 Australia MAGs with valid features ({holdout_mask_valid.sum()}) — evaluation is underpowered.")

if (~holdout_mask & valid_mask).sum() > 0 and holdout_mask_valid.sum() > 0:
    holdout_result = geographic_holdout_eval(
        df=df[valid_mask].reset_index(drop=True),
        holdout_mask=holdout_mask[valid_mask],
        feature_cols=MAG_ALL_DENSITY,
        target_col=TARGET,
    )
    print("Holdout results:", holdout_result)

    if not np.isnan(m1_cv_rmse):
        ratio = holdout_vs_cv_ratio(holdout_result['rmse_holdout'], m1_cv_rmse)
        print(f"Holdout/CV RMSE ratio: {ratio:.3f}")
    else:
        ratio = np.nan

    holdout_result['cv_rmse_m1'] = m1_cv_rmse
    holdout_result['holdout_cv_ratio'] = ratio
    holdout_result['holdout_region'] = 'Australia'

    with open(DATA_DIR / 'mgnify_geographic_holdout.csv', 'w') as f:
        pd.DataFrame([holdout_result]).to_csv(f, index=False)
else:
    print("Not enough samples for holdout evaluation.")

Holdout results: {'rmse_holdout': 0.02522581575347143, 'r2_holdout': -0.3747292932954125, 'n_holdout': 97, 'n_train': 7876}
Holdout/CV RMSE ratio: 0.655


In [6]:
# Map of holdout regions
if valid_mask.sum() > 0:
    fig, ax = plt.subplots(figsize=(FIGW['2col'], ROW_H))
    train_mask = ~holdout_mask & valid_mask
    if train_mask.sum() > 0:
        ax.scatter(
            df.loc[train_mask, 'longitude'], df.loc[train_mask, 'latitude'],
            s=5, c=PALETTE[0], alpha=0.4, label='Training MAGs'
        )
    if holdout_mask_valid.sum() > 0:
        ax.scatter(
            df.loc[holdout_mask_valid, 'longitude'], df.loc[holdout_mask_valid, 'latitude'],
            s=10, c=PALETTE[1], alpha=0.7, label='Holdout (Australia)'
        )
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_title('Geographic holdout — Australia (MGnify)')
    ax.legend()
    grid_h(ax)
    save(fig, FIG_DIR / 'nb03b_holdout_map')